# Parte 1: KNN

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [8]:
df = pd.read_csv('wine-clustering.csv')
df

,Alcohol,Malic_Acid,Ash,Ash_Alcanity,Magnesium,Total_Phenols,Flavanoids,Nonflavanoid_Phenols,Proanthocyanins,Color_Intensity,Hue,OD280,Proline
0,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735
...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740
174,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750
175,13.27,4.28,2.26,20.0,120,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835
176,13.17,2.59,2.37,20.0,120,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840


In [9]:
# Eliminamos filas con valores nulos (por precaución)
df = df.dropna()

In [11]:
# Variables predictoras 
X = df.drop(columns=['Alcohol'])
y = df['Alcohol']

In [13]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [10]:
# Definir punto a relacionar
test_data = pd.DataFrame({
    'Alcohol': [14],'Malic_Acid': [2],'Ash': [2.5],'Ash_Alcanity': [16],'Magnesium': [115],
    'Total_Phenols': [3],'Flavanoids': [2.5],'Nonflavanoid_Phenols': [0.4],
    'Proanthocyanins': [2],'Color_Intensity': [9],'Hue': [1],'OD280': [3.5],'Proline': [800]
})
test_data

,Alcohol,Malic_Acid,Ash,Ash_Alcanity,Magnesium,Total_Phenols,Flavanoids,Nonflavanoid_Phenols,Proanthocyanins,Color_Intensity,Hue,OD280,Proline
0,14,2,2.5,16,115,3,2.5,0.4,2,9,1,3.5,800


In [23]:
# Escalamos el test_data con el mismo scaler (sin incluir la columna Alcohol)
test_scaled = scaler.transform(test_data.drop(columns=['Alcohol']))

In [24]:
# Entrenamos el modelo KNN (para encontrar vecinos)
knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
knn.fit(X_scaled)

NearestNeighbors(metric='euclidean')

In [25]:
# Encontramos los 5 vecinos más cercanos
distances, indices = knn.kneighbors(test_scaled)

In [29]:
# Obtenemos los valores de Alcohol de los 5 vecinos
neighbor_alcohols = y.iloc[indices[0]]

In [30]:
# Calculamos el promedio
average_alcohol = neighbor_alcohols.mean()

In [28]:
print("Valores de Alcohol de los 5 vecinos más cercanos:")
print(neighbor_alcohols.values)
print("\nPromedio estimado de Alcohol:", round(average_alcohol, 3))

Valores de Alcohol de los 5 vecinos más cercanos:
[13.56 14.22 13.94 14.23 14.06]

Promedio estimado de Alcohol: 14.002


# Parte 2: Market Basket

In [39]:
my_basket = [['bread', 'butter', 'wine', 'bananas', 'coffee', 'carrots'],
    ['tomatoes', 'onions', 'cheese', 'milk', 'potatoes'],
    ['beer', 'chips', 'asparagus', 'salsa', 'milk', 'apples'],
    ['olive oil', 'bread', 'butter', 'tomatoes', 'steak', 'carrots'],
    ['tomatoes', 'onions', 'chips', 'wine', 'ketchup', 'orange juice'],
    ['bread', 'butter', 'beer', 'chips', 'milk'],
    ['butter', 'tomatoes', 'carrots', 'coffee', 'sugar'],
    ['tomatoes', 'onions', 'cheese', 'milk', 'potatoes'],
    ['bread', 'butter', 'ketchup', 'coffee', 'chicken wings'],
    ['butter', 'beer', 'chips', 'asparagus', 'apples'],
    ['tomatoes', 'onion', 'beer', 'chips', 'milk', 'coffee']]

In [41]:
# Transformación a formato binario (one-hot encoding)
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
te = TransactionEncoder()
te_ary = te.fit(my_basket).transform(my_basket)
df = pd.DataFrame(te_ary, columns=te.columns_)

In [42]:
# Aplicar el algoritmo Apriori
# min_support = 0.2 → el producto o conjunto aparece al menos en 20% de las transacciones
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)

In [43]:
# Generar reglas de asociación
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

In [44]:
# Ordenar las reglas por 'lift' (mayor asociación)
rules = rules.sort_values(by='lift', ascending=False)

In [45]:
# Mostrar resultados
print("🛒 Reglas de Asociación Detectadas (Top 10):")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

🛒 Reglas de Asociación Detectadas (Top 10):
      antecedents    consequents   support  confidence      lift
17  (chips, milk)         (beer)  0.272727    1.000000  2.750000
20         (beer)  (chips, milk)  0.272727    0.750000  2.750000
18   (beer, milk)        (chips)  0.272727    1.000000  2.200000
19        (chips)   (beer, milk)  0.272727    0.600000  2.200000
0         (chips)         (beer)  0.363636    0.800000  2.200000
1          (beer)        (chips)  0.363636    1.000000  2.200000
6       (carrots)       (butter)  0.272727    1.000000  1.833333
7        (butter)      (carrots)  0.272727    0.500000  1.833333
4         (bread)       (butter)  0.363636    1.000000  1.833333
5        (butter)        (bread)  0.363636    0.666667  1.833333
